### Set Up Environment

In [2]:
!python3 -m venv venv
!source venv/bin/activate

In [3]:
!pip3 install "google-cloud-bigquery>=3.17"
!pip3 install "google-cloud-aiplatform>=1.38"
!pip3 install "pandas>=2.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 2.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.2/230.2 kB 1.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 kB 2.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 2.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.3/140.3 kB 3.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.8/186.8 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 kB 2.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.9/120

In [7]:
import vertexai
from google.cloud import bigquery
import pandas as pd
import random
from vertexai.preview import generative_models
from vertexai.preview.generative_models import GenerativeModel, GenerationResponse
import json

#project_id = "ai-sandbox-sw"
project_id = "uk-bh-experiments-argolis"
dataset_id = "mstudy"
target_table = "target"
source_tables = ["source-uipetmis","source-uispet"]

### Get Source and target Schemas from BigQuery

To reproduce yourself, use the .csvs in this directory and upload to BigQuery in your own Project. Do not use the provided spreadsheets directly as I have done some (minimal) pre-processing on the spreadsheets

In [8]:
client = bigquery.Client(project=project_id)

target_query = f"""
    SELECT *  

    FROM `{project_id}.{dataset_id}.{target_table}`
"""

target_df = client.query(target_query).to_dataframe()

reference_df = target_df.copy() #To check Gemini's responses actually exist in the Target schema (halucination check)

reference_df = reference_df.rename(columns={
    "Tranche": "Target_Tranche",
    "Level_1": "Target_Level_1",
    "Level_2": "Target_Level_2",
    "Level_3": "Target_Level_3",
    "Level_4": "Target_Level_4",
    "Attribute": "Target_Attribute",
    "Complex_Type": "Target_Complex_Type",
    "Description": "Target_Description",
    "Mandatory__": "Target_Mandatory__",
    "Data_Type": "Target_Data_Type",
    "Accepted_Values": "Target_Accepted_Values",
    "Validation": "Target_Validation",
    "Drop_Down_Metaval": "Target_Drop_Down_Metaval"
})

target_df.head(5)

NotFound: 404 Not found: Dataset uk-bh-experiments-argolis:mstudy was not found in location US; reason: notFound, message: Not found: Dataset uk-bh-experiments-argolis:mstudy was not found in location US

Location: US
Job ID: d051e817-c984-4bb4-bc26-f470cb901ddd


In [ ]:
source_query = f"""
    SELECT *  
    FROM `{project_id}.{dataset_id}.source*`
"""
source_df = client.query(source_query).to_dataframe()
source_df.head(5)

### Helper functions to manipulate and format data

In [ ]:
def create_target_df_groups(df, max_level):
    """Groups a DataFrame by nested schema paths up to a specified level.

    Args:
        df: The DataFrame to group.
        max_level: The maximum nesting level to group by (1, 2, 3, or 4).

    Returns:
        A dictionary of DataFrames, where keys are the nested paths, and
        values are DataFrames containing fields sharing that path. 
    """

    grouped_dfs = {}
    levels = ['Tranche'] + [f'Level_{i}' for i in range(1, max_level + 1)] 

    for _, row in df.iterrows():
        path = '.'.join(row[col] for col in levels if row[col] != 'n/a')
        if path not in grouped_dfs:
            grouped_dfs[path] = pd.DataFrame(columns=df.columns)  
        grouped_dfs[path] = pd.concat([grouped_dfs[path], row.to_frame().T], ignore_index=True)

    return grouped_dfs

def create_source_df_groups(df):
    """Groups a DataFrame by nested schema paths up to a specified level.

    Args:
        df: The DataFrame to group.

    Returns:
        A dictionary of DataFrames, where keys are the nested paths, and
        values are DataFrames containing fields sharing that path. 
    """

    grouped_dfs = {}
    levels = ['SchemaName', 'TableName'] 

    for _, row in df.iterrows():
        path = '.'.join(row[col] for col in levels)
        if path not in grouped_dfs:
            grouped_dfs[path] = pd.DataFrame(columns=df.columns)  
        grouped_dfs[path] = pd.concat([grouped_dfs[path], row.to_frame().T], ignore_index=True)

    return grouped_dfs

def dataframe_to_string(df):
    """Converts a DataFrame to a string with column names and row values.

    Args:
        df: The pandas DataFrame to convert.

    Returns:
        A string representation of the DataFrame.
    """

    output = f"Column Names: {', '.join(df.columns)}\n"  # Header with column names

    for _, row in df.iterrows():
        row_string = ', '.join(str(value) for value in row)
        output += f"Row: {row_string}\n"

    return output

def find_dataframes_with_string(dataframes, search_string):
  """Searches for a string in any column across a list of DataFrames.

  Args:
      dataframes: A list of pandas DataFrames.
      search_string: The string to search for.

  Returns:
      A dictionary where keys are DataFrame indices in the original list,
      and values are the corresponding DataFrames containing the search_string.
  """

  matching_dataframes = {}
  for i, df in enumerate(dataframes):
    # Search for the string in any column using vectorized operations
    if df.map(lambda x: search_string in str(x)).any().any():
      df = df.reset_index(drop=True)
      matching_dataframes[i] = df

  return matching_dataframes

### Prepare the subdivided groups for the source and destination schemas

This is required so we can come in below the maximum token size for Gemini
(32K input, 2K output) https://ai.google.dev/models/gemini#model_variations

In [ ]:
target_df_groups = create_target_df_groups(target_df, max_level=3)
print("Number of target schema dataframe groupings: ", len(target_df_groups))

target_string_groups = []
for path, target_df in target_df_groups.items():
    target_string = dataframe_to_string(target_df)
    target_string_groups.append(target_string)

print("Number of target schema string groupings: ", len(target_string_groups))
random_number_in_target_range = random.randint(0, len(target_string_groups)-1)
print("showing randomly chosen target string group, ",random_number_in_target_range)
random_target_string_group = target_string_groups[random_number_in_target_range]
print(random_target_string_group)


print("*************")

source_df_groups = create_source_df_groups(source_df)
print("Number of source schema dataframe groupings: ", len(source_df_groups))

source_string_groups = []
for path, source_group_df in source_df_groups.items():
    source_group_sting = dataframe_to_string(source_group_df)
    source_string_groups.append(source_group_sting)

print("Number of source schema string groupings: ", len(source_string_groups))
# random_number_in_source_range = random.randint(0, len(source_string_groups)-1)
random_number_in_source_range = 5
print("showing randomly chosen source string group, ",random_number_in_source_range)
random_source_string_group = source_string_groups[random_number_in_source_range]
print(random_source_string_group)

In [ ]:
# # For testing before iterating over all source:target combinations, we manually select three source and one target groups.

# target_filter = 'Client Personal Data'
# filtered_target_strings = [string for string in target_string_groups if target_filter in string]

# target_df_groups_list = list(target_df_groups.values())
# filtered_target_dfs = find_dataframes_with_string(target_df_groups_list, target_filter)

# print("Number of filtered target strings: ", len(filtered_target_strings),"\n")
# for filtered_target_string in filtered_target_strings:
#     print(filtered_target_string)

# print("Number of filtered target dfs: ", len(filtered_target_dfs),"\n")
# print(type(filtered_target_dfs))
# for df_index, df in filtered_target_dfs.items():
#     print(df.head())

# source_filters = ['affinity_owner','customer']
# filtered_source_strings = [string for string in source_string_groups if any(substring in string for substring in source_filters)]

# print("Number of filtered source strings: ", len(filtered_source_strings),"\n")
# for filtered_source_string in filtered_source_strings:
#     print(filtered_source_string)


### Helper functions used post LLM-response

In [ ]:
def parse_function_call(function_call):
    """Parses a FunctionCall object, adds a description, and returns a JSON-compatible dictionary.

    Args:
        function_call: The FunctionCall object to parse.

    Returns:
        A dictionary containing the function name, arguments, and description.
    """

    result = {
        "function_name": function_call.name,
        "arguments": {},
    }
    for key, value in function_call.args.items():
        result["arguments"][key] = value

    return result

def convert_dict_to_list_of_dicts(dict):
    """Converts a dictionary of lists and strings to a list of flat dictionaries.

    Args:
        data: The input dictionary containing lists and strings.

    Returns:
        A list of dictionaries, where each dictionary represents a  
        combination of elements from the input lists.
    """
    
    list_of_attribute_dicts = []
    string_keys = []
    list_keys = []

    
    for key, value in dict.items() :
        if isinstance(value, str):
            string_keys.append(key)
        else:
            list_keys.append(key)       
    
    for i in range (len(dict[list_keys[0]])):
        new_dict = {}
        
        for key in list_keys:
            new_dict[key] = dict[key][i]
        for key in string_keys:
            new_dict[key] = dict[key]

        list_of_attribute_dicts.append(new_dict)

    return list_of_attribute_dicts

def match_mappings_to_target_schema(attributes, reference_df):    
    
    print(f"Looking at: ", attributes)
    condition1 = reference_df['Target_Tranche'] == attributes['Target_Tranche']
    condition2 = reference_df['Target_Level_1'] == attributes['Target_Level_1']
    condition3 = reference_df['Target_Level_2'] == attributes['Target_Level_2']
    condition4 = reference_df['Target_Level_3'] == attributes['Target_Level_3']
    condition5 = reference_df['Target_Level_4'] == attributes['Target_Level_4']
    condition6 = reference_df['Target_Attribute'] == attributes['Target_Attribute']

    matched_rows = reference_df[condition1 & condition2 &condition3 & condition4 & condition5 & condition6]

    if not matched_rows.empty:
        print(len(matched_rows), " Matched row(s) found against the reference! On row(s) with Unique_Ref:",matched_rows['Unique_Ref'].tolist() )
    else:
        print('match not found')
        
    for col in attributes:
        if col not in matched_rows.columns:
            matched_rows.loc[:, col] = attributes[col]  # Use .loc

    return matched_rows

### Prepare Gemini

- First we create the FunctionDeclaration. This helps us get a more structured and consistent output from the LLM which is helpful for use cases such as this when we are dealing with structured data. See [Function Calling](https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/function-calling)
- Then we prepare the prompt to send to Gemini. The prompt is intentionally very verbose and repetitive, as well as directly refering to the declared function. Further optimisations and improvements could be made if spent tuning the prompt.

In [ ]:
model = GenerativeModel("gemini-pro")

get_mappings_func = generative_models.FunctionDeclaration(
  name="get_mappings",
  description="Get the mappings of source schema feilds for a given target schema field. For when provided with 1 field in a target schema and multiple fields in a source schema, this function provides the mappings as a confidence level for source field to the target field. The target fields should be recorded as indexes in the arrays for Source_Column_Names, Source_TableNames and Source_SchemaNames, where the same index of each array describes the source field details for that row. for example the values in Source_Column_Names[1] Source_TableNames[1]Source_SchemaNames[1] together describe the attributes of the same source field, and the value in Confidence_Levels[1] describe the confidence level out of 10 for mapping this source field to the target field",
  parameters={
      "type": "object",
      "properties": {
          "Target_Tranche": {
              "type": "string",
              "description": "The Tranche of this field in the target schema",
              "enum": ["CLAIMS","CLIENT","POLICY"]
          },
          "Target_Level_1": {
              "type": "string",
              "description": "The Level_1 nesting of this field in the target schema",
              "enum": ["Configuration","Client","Policy","Motor Policy","Household Policy","Pet"]
          },
          "Target_Level_2": {
              "type": "string",
              "description": "The Level_2 nesting of this field in the target schema"
          },
         "Target_Level_3": {
              "type": "string",
              "description": "The Level_3 nesting of this field in the target schema",
          },
          "Target_Level_4": {
              "type": "string",
              "description": "The Level_4 nesting of this field in the target schema"
          },
          "Target_Attribute": {
              "type": "string",
              "description": "The Attribute nesting of this field in the target schema"
          },
          "Source_SchemaNames": {
              "type": "array",
              "description": "The array of values for the field SchemaNames in the source schema that could map to the target schema with a likleyhood based on the corresponding value of Confidence_Level property at the same index.",
              "items" : {
                    "type": "string"
              },
              "example": ["dbo","dbo","dbo"]
          },
          "Source_TableNames": {
              "type": "array",
              "description": "The array of values for the field TableNames in the source schema that could map to the target schema with a likleyhood based on the corresponding value of Confidence_Level property at the same index.",
              "items" : {
                    "type": "string"
              },
              "example": ["user_message","CDLPolicyImportWrk","clinic"]
          },
          "Source_Column_Names": {
              "type": "array",
              "description": "The array of values for the field Column_Names in the source schema that could map to the target schema with a likleyhood based on the corresponding value of Confidence_Level property at the same index.",
              "items" : {
                "type": "string"
              },
              "example": ["message_id","PremiumPrice","area_id"]
          },
          "Confidence_Levels": {
              "type": "array",
              "description": "The array of values for the confidence level out of 10 for the mappings of the source sschema fields at the corresponding index. For example the array of ['3','8','2'] would mean a confidence level of 3/10 for the source>target schema mapping for the source field represented by the values in the first indexes of the Source_Column_Names, Source_TableNames and Source_SchemaNames arrays, a confidence level of 8/10 for the the source>target schema mapping for the source field represented by the values in the second indexes of the Source_Column_Names, Source_TableNames and Source_SchemaNames arrays, and so on for each of the indexes of the arrays.",
              "items" : {
                    "type": "string",
                    "enum": ["0","1","2","3","4","5","6","7","8","9","10"]
              },
              "example": ["1","2","3"]
          }
      },
      "required": [
          "Target_Tranche", "Target_Level_1", "Target_Level_2", "Target_Level_3", "Target_Level_4", "Target_Attribute", "Source_SchemaName", "Source_TableName", "Source_Column_Name", "Confidence_Level"
      ]
  },
)

get_mappings_tool = generative_models.Tool(
  function_declarations=[get_mappings_func]
)

### Test case 1: Testing with just a single target field and small group of source fields

In [ ]:
# For testing before iterating over all source:target combinations, we manually select three source and one target groups.

test_target_df_row = target_df.iloc[[5]]
test_target_string_row = dataframe_to_string(test_target_df_row)
test_target_df_row.head()

### Test case 1: Prepare LLM

- First we create the FunctionDeclaration. This helps us get a more structured and consistent output from the LLM which is helpful for use cases such as this when we are dealing with structured data. See [Function Calling](https://cloud.google.com/vertex-ai/docs/generative-ai/multimodal/function-calling)
- Then we prepare the prompt to send to Gemini. The prompt is intentionally very verbose and repetitive, as well as directly refering to the declared function. Further optimisations and improvements could be made if spent tuning the prompt.

In [ ]:
prompt = f"""You are Data Engineer working for an insurance company. As part of a data migration project you need to assist with mapping fields in a source data schema fields in a target data schema.
The source and desination schemas are both complex and nested.
You will be shown 1 field in the target schema and multiple fields in the source schema.
The mappings will not be exactly one to one.
Instead of providing a one-to-one mapping for a single source schema to a single destiation schema, you will be asked to provide a confidence rating for how well you think each of the fields for the source schemas you see will map to the field for the target schema.

The field from the target schema is described here:
{test_target_string_row}

The fields taken from the source schema are described here:
{random_source_string_group}

Based on what you can see, I want you to provide a confidence level for whether any of the fields in the source schema map to the field of the target schema.
The confidence level is a number between 0 and 10 where 0 is very unlikely and 10 is a very strong match.

Please use the get_mappings_tool to structure your response, where each seperate field of the target schema should be described in the same index of the arrays for the Confidence_Levels, Source_Column_Names, Source_TableNames and Source_SchemaNames properties.

As an example:
assuming a field the target schema is:
Column Names: Unique_Ref, Tranche, Level_1, Level_2, Level_3, Level_4, Complex_Type, Attribute, Description, Mandatory__, Data_Type, Accepted_Values, Validation, Drop_Down_Metaval
Row: 382, POLICY, Policy, quotesHubLifestyleFactor, n/a, n/a, quotesHubLifestyleFactor, code, The code identifying the Lifestyle factor, Mandatory, string (30), None, None, <NA>

and the fields for the source schema to provide a confidence level mapping for are:
Column Names: SchemaName, TableName, Column_Name, Data_type, Max_Length, precision, scale, is_nullable
Row: dbo, user_message, message_id, int, 4, 10, 0, 0
Row: dbo, CDLPolicyImportWrk, PremiumPrice, money, 8, 19, 4
Row: dbo, clinic, area_id, int, 4, 10, 0

You should structure the response with the get_mappings tool as follows:
Confidence_Levels = ["X", "Y", "Z"]
Source_SchemaNames = ["dbo","dbo","dbo"]
Source_TableNames = ["user_message","CDLPolicyImportWrk","clinic"]
Source_Column_Names = ["message_id","PremiumPrice","area_id"]

Where X represents your confidence out of 10 for how the sorce field dbo.user_message.message_id maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor
Where Y represents your confidence out of 10 for how the sorce field dbo.CDLPolicyImportWrk.PremiumPrice maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor
Where Z represents your confidence out of 10 for how the sorce field dbo.clinic.area_id maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor

IT IS VERY IMPORTANT THAT IF THE LENGTHS OF THE ARRAYS YOU CREATE FOR THE PARAMETERS Source_Column_Names, Source_TableNames, Source_SchemaNames AND Confidence_Levels ARE EXACTLY THE SAME AS THE NUMBER OF SOURCE FIELDS YOU ARE SHOWN.
YOU MUST CREATE EXACTLY ONE CONFIDENCE LEVEL MAPPING FROM EACH SOURCE FIELD TO THE TARGET FIELD, AND FOLLOW THE INSTRUCTIONS EXACTLY FOR HOW TO STRUCTURE THIS INFORMATION INTO THE get_mappings TOOL.

There is a strong chance that none of the fields provide to you will match well - remember I am only sending you a small portion of the target schema and there may be other fields that you haven't seen which map much better.
"""

print("test_target_string_row\n", test_target_string_row)
print("random_source_string_group\n", random_source_string_group)


model_response = model.generate_content(
    prompt,
    generation_config={"temperature": 0},
    tools=[get_mappings_tool],
)

print("model_response\n", model_response)



### Test case 1: Prepare response for combining schemas and uploading to BigQuery

In [ ]:
model_function_call_json = parse_function_call(model_response.candidates[0].content.parts[0].function_call)
print(type(model_function_call_json))
print(model_function_call_json)

attributes_dict = model_function_call_json["arguments"]

list_of_attribute_dicts = convert_dict_to_list_of_dicts(attributes_dict) #flattens the response from the LLM so we now have a list containing 1 dict per source field > destination field mapping

### Test case 1: Iterate through each mapping from LLM to check correct mappings, then upload successful mappings to BigQuery

In [ ]:
mapped_df = pd.DataFrame() #The final output to BigQuery

for i, attribute_dict in enumerate(list_of_attribute_dicts):    
    print(f"[{i+1}/{len(list_of_attribute_dicts)},] Updating mapped_df following mapping excercise by Gemini. Number of source fields to review:", len(list_of_attribute_dicts))
    temp_matched_df = match_mappings_to_target_schema(attribute_dict, reference_df)
    
    for col in attribute_dict:
        if col not in temp_matched_df.columns:
            temp_matched_df.loc[col] = attribute_dict[col]

    mapped_df = pd.concat([mapped_df, temp_matched_df], ignore_index=True)
    print(f"source field {i+1}/{len(list_of_attribute_dicts)} successfully reviewed")

dataset_ref = client.dataset(dataset_id)
table_ref = dataset_ref.table("mapped-test1")

# Create a job configuration
job_config = bigquery.LoadJobConfig(
    schema=[],
    write_disposition="WRITE_APPEND", 
)

job = client.load_table_from_dataframe(mapped_df, table_ref, job_config=job_config)  
job.result()  # Wait for job completion


### Test case 2: Iterate over all source fields groups (~500), for a single target field 

- expect errors due to inconsistent structure back from LLLM . We will use error handling to just skip failling fields

In [ ]:
def chop_source_df_groups(source_df_groups, max_rows_per_group):
    """Chops source dataframe groups into smaller groups with a specified max number of rows.

    Args:
        source_df_groups: The dictionary of source dataframe groups.
        max_rows_per_group: The maximum number of rows allowed in each group.

    Returns:
        A modified dictionary of source dataframe groups with smaller groups.
    """
    chopped_source_df_groups = {}

    for path, source_group_df in source_df_groups.items():
        # Check if the group needs to be chopped
        if len(source_group_df) <= max_rows_per_group:
            chopped_source_df_groups[path] = source_group_df
        else:
            # Split the group into smaller groups with a maximum of max_rows_per_group rows
            num_subgroups = len(source_group_df) // max_rows_per_group
            remainder = len(source_group_df) % max_rows_per_group

            for i in range(num_subgroups):
                start_idx = i * max_rows_per_group
                end_idx = (i + 1) * max_rows_per_group
                sub_df = source_group_df.iloc[start_idx:end_idx]
                chopped_source_df_groups[f"{path}_subgroup_{i+1}"] = sub_df

            # Add the remainder as a separate subgroup
            if remainder > 0:
                sub_df = source_group_df.iloc[-remainder:]
                chopped_source_df_groups[f"{path}_subgroup_{num_subgroups+1}"] = sub_df

    return chopped_source_df_groups


#chop up the source string groups further (seems to help with LLM accuracy):
print("*************")
chopped_source_df_groups = chop_source_df_groups(source_df_groups, 4)
print("Number of chopped source schema dataframe groupings: ", len(chopped_source_df_groups))

chopped_source_string_groups = []
for path, chopped_source_group_df in chopped_source_df_groups.items():
    chopped_source_group_sting = dataframe_to_string(chopped_source_group_df)
    chopped_source_string_groups.append(chopped_source_group_sting)
print("Number of chopped source schema string groupings: ", len(chopped_source_string_groups))
random_source_string_group = chopped_source_string_groups[7]
print(random_source_string_group)

In [ ]:
dataset_ref = client.dataset(dataset_id)
table_ref_test2 = dataset_ref.table("mapped-test2")

# Create a job configuration
job_config = bigquery.LoadJobConfig(
    schema=[],
    write_disposition="WRITE_APPEND", 
)




for j, source_string_group in enumerate(chopped_source_string_groups):

    print("************************************")
    print(f"Attempting source field group {j}...")
    print("************************************")
    
    prompt = f"""You are Data Engineer working for an insurance company. As part of a data migration project you need to assist with mapping fields in a source data schema fields in a target data schema.
    The source and desination schemas are both complex and nested.
    You will be shown 1 field in the target schema and multiple fields in the source schema.
    The mappings will not be exactly one to one.
    Instead of providing a one-to-one mapping for a single source schema to a single destiation schema, you will be asked to provide a confidence rating for how well you think each of the fields for the source schemas you see will map to the field for the target schema.

    The field from the target schema is described here:
    {test_target_string_row}

    The fields taken from the source schema are described here:
    {source_string_group}

    Based on what you can see, I want you to provide a confidence level for whether any of the fields in the source schema map to the field of the target schema.
    The confidence level is a number between 0 and 10 where 0 is very unlikely and 10 is a very strong match.

    Please use the get_mappings_tool to structure your response, where each seperate field of the target schema should be described in the same index of the arrays for the Confidence_Levels, Source_Column_Names, Source_TableNames and Source_SchemaNames properties.

    As an example:
    assuming a field the target schema is:
    Column Names: Unique_Ref, Tranche, Level_1, Level_2, Level_3, Level_4, Complex_Type, Attribute, Description, Mandatory__, Data_Type, Accepted_Values, Validation, Drop_Down_Metaval
    Row: 382, POLICY, Policy, quotesHubLifestyleFactor, n/a, n/a, quotesHubLifestyleFactor, code, The code identifying the Lifestyle factor, Mandatory, string (30), None, None, <NA>

    and the fields for the source schema to provide a confidence level mapping for are:
    Column Names: SchemaName, TableName, Column_Name, Data_type, Max_Length, precision, scale, is_nullable
    Row: dbo, user_message, message_id, int, 4, 10, 0, 0
    Row: dbo, CDLPolicyImportWrk, PremiumPrice, money, 8, 19, 4
    Row: dbo, clinic, area_id, int, 4, 10, 0

    You should structure the response with the get_mappings tool as follows:
    Confidence_Levels = ["X", "Y", "Z"]
    Source_SchemaNames = ["dbo","dbo","dbo"]
    Source_TableNames = ["user_message","CDLPolicyImportWrk","clinic"]
    Source_Column_Names = ["message_id","PremiumPrice","area_id"]

    Where X represents your confidence out of 10 for how the sorce field dbo.user_message.message_id maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor
    Where Y represents your confidence out of 10 for how the sorce field dbo.CDLPolicyImportWrk.PremiumPrice maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor
    Where Z represents your confidence out of 10 for how the sorce field dbo.clinic.area_id maps to the target field POLICY.Policy.quotesHubLifestyleFactor.quotesHubLifestyleFactor

    IT IS VERY IMPORTANT THAT IF THE LENGTHS OF THE ARRAYS YOU CREATE FOR THE PARAMETERS Source_Column_Names, Source_TableNames, Source_SchemaNames AND Confidence_Levels ARE EXACTLY THE SAME AS THE NUMBER OF SOURCE FIELDS YOU ARE SHOWN.
    YOU MUST CREATE EXACTLY ONE CONFIDENCE LEVEL MAPPING FROM EACH SOURCE FIELD TO THE TARGET FIELD, AND FOLLOW THE INSTRUCTIONS EXACTLY FOR HOW TO STRUCTURE THIS INFORMATION INTO THE get_mappings TOOL.

    There is a strong chance that none of the fields provide to you will match well - remember I am only sending you a small portion of the target schema and there may be other fields that you haven't seen which map much better.
    """

    print("source_string_group\n", source_string_group)


    model_response_test2 = model.generate_content(
        prompt,
        generation_config={"temperature": 0},
        tools=[get_mappings_tool],
    )

    try:
        #Prepare model responses for mapping check
        model_function_call_json_test2 = parse_function_call(model_response_test2.candidates[0].content.parts[0].function_call)
        attributes_dict_test2 = model_function_call_json_test2["arguments"]
        list_of_attribute_dicts_test2 = convert_dict_to_list_of_dicts(attributes_dict_test2) #flattens the response from the LLM so we now have a list containing 1 dict per source field > destination field mapping

        mapped_df_test2 = pd.DataFrame() #The final output to BigQuery

        for i, attribute_dict_test2 in enumerate(list_of_attribute_dicts_test2):    
            print(f"[{i+1}/{len(list_of_attribute_dicts_test2)},] Updating mapped_df_test2 following mapping excercise by Gemini. Number of source fields to review:", len(list_of_attribute_dicts_test2))
            print(f"this field: {attribute_dict_test2}")
            temp_matched_df = match_mappings_to_target_schema(attribute_dict_test2, reference_df)
            
            for col in attribute_dict:
                if col not in temp_matched_df.columns:
                    temp_matched_df.loc[col] = attribute_dict[col]

            mapped_df_test2 = pd.concat([mapped_df_test2, temp_matched_df], ignore_index=True)

        job = client.load_table_from_dataframe(mapped_df_test2, table_ref_test2, job_config=job_config)  
        job.result()  # Wait for job completion

    except Exception as error:
        print("FAILED!!! ", error)
        print(source_string_group)
        print("UNABLE TO SUCCESSFULLY MAP THE ABOVE TARGET FIELDS")
        print("SKIPPING AND MOVING TO NEXT TARGET FIELD GROUPS")


